# Lab 6: Point Estimation and Mixture Priors

**Phân tích dữ liệu Bayesian - IUH**

Lab 6 đặt người học trước hai kiểu do dự khác nhau. Kiểu thứ nhất là do dự sau khi đã có posterior: nên đọc nó bằng mean, median, MAP hay một xác suất vượt ngưỡng? Kiểu thứ hai đến sớm hơn, ngay từ prior: nếu trước dữ liệu ta chưa chắc chỉ có một niềm tin nền duy nhất, thì có nên để nhiều khả năng cùng tồn tại hay không?

Chính vì vậy, lab này nối hai chủ đề tưởng xa nhau mà rất hợp nhau: quyết định Bayesian và mixture priors. Một bên hỏi cách đọc posterior. Bên kia hỏi cách xây prior khi tri thức ban đầu không thuần nhất.


In [ ]:
# Lab 6 cần cả hàm Beta/Gamma lẫn log-density để cập nhật mixture priors ổn định về số học.
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import betaln, gammaln

plt.rcParams["figure.figsize"] = (10, 5)


## 1. Khi một posterior có nhiều cách đọc và một prior có nhiều tiếng nói

Ở phần đầu, bài toán trông quen thuộc: posterior đã có rồi, giờ chỉ còn chọn một điểm đại diện. Nhưng đây lại là nơi sinh viên dễ mắc sai lầm nhất, vì rất dễ xem mean, median và MAP như ba cách tính khác nhau của cùng một đáp án, thay vì ba quyết định khác nhau dưới ba tiêu chuẩn mất mát khác nhau.

Sang phần sau, mức độ bất định được đẩy lùi thêm một tầng nữa. Không chỉ posterior mới có nhiều cách đọc; ngay cả prior cũng có thể là một hỗn hợp của nhiều khả năng nền. Khi đó, dữ liệu không chỉ cập nhật tham số mà còn tái phân bố độ tin cậy giữa các kịch bản prior cạnh tranh nhau.


## 2. Bài 1 - Posterior và các ước lượng điểm

Với prior $f(x)=2x$ trên $[0,1]$ và likelihood $f(y\mid x)=4xy$, quan sát $y=\frac{1}{5}$ chỉ tạo ra một hằng số nhân theo $x$. Vì vậy posterior tỉ lệ với $x^2$, tức là Beta$(3,1)$.

### Ý nghĩa của bài tập

Bài mở đầu này nhấn mạnh rằng posterior là một phân phối, không phải một đáp án điểm duy nhất. Mean, median và MAP chỉ là ba cách đọc khác nhau của cùng một posterior dưới ba tiêu chuẩn mất mát khác nhau.

Đây là bước rất quan trọng để người học chuyển từ tư duy “tính ra số đúng” sang tư duy quyết định Bayesian. Một con số chỉ có ý nghĩa khi ta nói rõ nó đang tối ưu cho mục tiêu nào.


### Đề bài nhắc lại

**Phần 1 - Bài 1.** Cho prior $X\sim 2x$ trên $[0,1]$, likelihood $Y\mid X=x\sim 4xy$ trên $0\le y\le 1$, và quan sát $y=1/5$. Hãy tìm các ước lượng điểm tốt nhất của hậu nghiệm theo ba hàm tổn thất: bình phương, trị tuyệt đối, và 0-1.


In [ ]:
# Với posterior Beta(3,1), ba ước lượng mean/median/MAP lần lượt minh họa ba tiêu chuẩn tổn thất khác nhau.
posterior = stats.beta(3, 1)
posterior_mean = posterior.mean()
posterior_median = posterior.ppf(0.5)
posterior_map = 1.0
print(f"Posterior mean (squared loss) = {posterior_mean:.4f}")
print(f"Posterior median (absolute loss) = {posterior_median:.4f}")
print(f"Posterior MAP (0-1 loss) = {posterior_map:.4f}")


## 3. Bài 2 - Thay kiểm định một phía bằng posterior probability

Đề gốc dùng ngôn ngữ kiểm định giả thuyết. Ở đây ta giữ nguyên câu hỏi thực chất nhưng viết lại theo tinh thần Bayesian: posterior có ủng hộ mạnh cho mệnh đề $x>0.7$ hay không?

### Ý nghĩa của bài tập

Bài này viết lại một câu hỏi quen thuộc của kiểm định giả thuyết bằng ngôn ngữ Bayesian trực tiếp hơn. Thay vì hỏi có “bác bỏ” một giả thuyết hay không, ta hỏi posterior dành bao nhiêu xác suất cho mệnh đề quan tâm.

Điều này giúp diễn giải kết quả trung thực và dễ hiểu hơn. Một posterior probability bằng 0.657 không phải là bằng chứng mạnh, nhưng nó nói rõ mức độ ủng hộ của dữ liệu theo đúng thang đo xác suất, thay vì ép kết luận vào khuôn nhị phân chấp nhận hoặc bác bỏ.


### Đề bài nhắc lại

**Phần 1 - Bài 2.** Với cùng thông tin như Bài 1, xét giả thuyết một phía
$$H_0: x \le 0.7,\qquad H_1: x < 0.7$$
và mức ý nghĩa $0.1$. Solution hiện diễn giải lại bài này theo posterior probability thay vì logic kiểm định nhị phân.


In [ ]:
# Thay ngôn ngữ “bác bỏ/chấp nhận” bằng posterior probability để giữ đúng tinh thần suy luận Bayesian.
posterior_prob_gt_07 = 1 - posterior.cdf(0.7)
print(f"P(x > 0.7 | y=1/5) = {posterior_prob_gt_07:.4f}")
print("If we require posterior probability > 0.9 to assert x > 0.7, evidence is insufficient.")


## 4. Bài 3 - Mixture of two normals

Mixture of two normals là bài đầu tiên trong lab nơi prior không còn mang một đỉnh niềm tin duy nhất. Thay vào đó, prior mã hóa hai khả năng nền khác nhau và cho phép người học biểu diễn bất định ở cấp độ mô hình, không chỉ ở cấp độ tham số.

### Ý nghĩa của bài tập

Ý nghĩa cốt lõi của bài là giúp sinh viên thấy có những lúc một prior đơn lẻ là quá nghèo để mô tả kiến thức ban đầu. Nếu trước dữ liệu ta thực sự phân vân giữa hai kịch bản, mixture prior là cách trung thực hơn để diễn đạt trạng thái hiểu biết đó.

Bài cũng mở đường cho trực giác về đa đỉnh, phân nhóm ẩn và heterogeneity trong các mô hình Bayesian hiện đại.


### Đề bài nhắc lại

**Phần 2 - Bài 1.** Cho prior hỗn hợp của $X$ gồm $45\%\,N(1.4,0.4^2)$ và $55\%\,N(2.1,0.5^2)$. Hãy xác định phân phối prior hỗn hợp của $X$, bao gồm kỳ vọng và phương sai.


In [ ]:
# Mixture prior hai chuẩn: tính mean và variance tổng thể trước khi nhìn dữ liệu để thấy prior đang mã hóa điều gì.
weights = np.array([0.45, 0.55])
means = np.array([1.4, 2.1])
variances = np.array([0.4**2, 0.5**2])
mixture_mean = np.sum(weights * means)
mixture_var = np.sum(weights * (variances + means**2)) - mixture_mean**2
print(f"Mixture prior mean = {mixture_mean:.4f}")
print(f"Mixture prior variance = {mixture_var:.6f}")


## 5. Bài 4 - Beta mixture posterior for Binomial data

Bài Beta mixture posterior cho thấy khi prior là hỗn hợp, dữ liệu không chỉ cập nhật tham số của từng thành phần mà còn cập nhật luôn độ tin cậy tương đối giữa các thành phần đó. Nói cách khác, dữ liệu đang “bỏ phiếu” xem kịch bản prior nào hợp lý hơn.

### Ý nghĩa của bài tập

Đây là một bước tiến rất quan trọng về mặt trực giác. Trong prior liên hợp thông thường, ta chỉ theo dõi một bộ tham số. Với mixture prior, bản thân cấu trúc posterior cũng giàu hơn vì nó mang thông tin về sự cạnh tranh giữa nhiều giả thuyết nền.

Bài này vì thế dạy sinh viên cách phân rã một posterior phức tạp thành hai tầng: posterior trong từng thành phần và posterior trên trọng số các thành phần.


### Đề bài nhắc lại

**Phần 2 - Bài 2.** Cho dữ liệu $n=50$, $x=36$, likelihood Binomial theo $\theta$, và prior hỗn hợp gồm $40\%\,\text{Beta}(20,30)$ với $60\%\,\text{Beta}(40,50)$. Hãy xác định posterior hỗn hợp của $\theta\mid x$.


In [ ]:
# Mixture Beta-Binomial: cập nhật từng thành phần trước, rồi mới chuẩn hóa lại trọng số posterior của hỗn hợp.
n, x_obs = 50, 36
prior_weights = np.array([0.4, 0.6])
components = [(20, 30), (40, 50)]
post_components = [(20 + x_obs, 30 + n - x_obs), (40 + x_obs, 50 + n - x_obs)]

log_weights = []
for w, (a, b) in zip(prior_weights, components):
    log_weights.append(math.log(w) + betaln(a + x_obs, b + n - x_obs) - betaln(a, b))
log_weights = np.array(log_weights)
normalized_weights = np.exp(log_weights - log_weights.max())
normalized_weights = normalized_weights / normalized_weights.sum()

print('Posterior component weights:', normalized_weights)
print('Posterior components:', post_components)

component_means = np.array([a / (a + b) for a, b in post_components])
component_vars = np.array([a * b / ((a + b)**2 * (a + b + 1)) for a, b in post_components])
posterior_mean = np.sum(normalized_weights * component_means)
posterior_var = np.sum(normalized_weights * (component_vars + component_means**2)) - posterior_mean**2
print(f"Mixture posterior mean = {posterior_mean:.4f}")
print(f"Mixture posterior variance = {posterior_var:.6f}")


## 6. Bài 5 - Gamma mixture posterior for Exponential data

Bài Gamma mixture cho dữ liệu Exponential lặp lại ý tưởng của Bài 4 nhưng trong bối cảnh tham số dương liên tục. Điều này giúp người học thấy mixture updating không phải một mẹo riêng của Beta-Binomial, mà là một khuôn suy luận có thể tái sử dụng ở nhiều họ phân phối khác nhau.

### Ý nghĩa của bài tập

Ý nghĩa lớn nhất của bài là xây sự linh hoạt trong tư duy mô hình hóa. Khi kiến thức ban đầu có nhiều nguồn hoặc nhiều kịch bản, ta không cần ép chúng vào một prior đơn; ta có thể để dữ liệu tự điều chỉnh trọng số giữa các khả năng cạnh tranh.

Đây cũng là cầu nối rất tốt sang các bài mixture models tổng quát hơn ở phần sau của course.


### Đề bài nhắc lại

**Phần 2 - Bài 3.** Cho mẫu kích thước $15$ từ phân phối mũ $X\mid \theta\sim \text{Exp}(\theta)$ với trung bình mẫu $\overline{x}=3$. Prior hỗn hợp cho $\theta$ là
$$0.6\times \text{Gamma}(7,8) + 0.4\times \text{Gamma}(10,9).$$
Hãy xác định các hệ số của các hậu nghiệm thành phần.


In [ ]:
# Mixture Gamma cho dữ liệu exponential: cùng một ý tưởng như trên nhưng với họ phân phối dương liên tục.
prior_weights_exp = np.array([0.6, 0.4])
components_exp = [(7, 8), (10, 9)]  # (shape, rate)
n = 15
xbar = 3
sum_x = n * xbar
post_components_exp = [(7 + n, 8 + sum_x), (10 + n, 9 + sum_x)]

log_weights_exp = []
for w, (a, b) in zip(prior_weights_exp, components_exp):
    log_weights_exp.append(math.log(w) + a * math.log(b) - gammaln(a) + gammaln(a + n) - (a + n) * math.log(b + sum_x))
log_weights_exp = np.array(log_weights_exp)
normalized_weights_exp = np.exp(log_weights_exp - log_weights_exp.max())
normalized_weights_exp = normalized_weights_exp / normalized_weights_exp.sum()

print('Posterior weights for Gamma mixture:', normalized_weights_exp)
print('Posterior Gamma components:', post_components_exp)


## 7. Model criticism and reflection

Mixture priors rất mạnh vì chúng cho phép ta nói “tôi không chắc prior nào đúng” thay vì cố ép mọi niềm tin về một phân phối duy nhất. Nhưng cái giá phải trả là posterior không còn tóm gọn bằng một cặp tham số đơn giản. Vì vậy, khi dùng mixture priors, ta nên báo cáo cả trọng số hậu nghiệm lẫn posterior của từng thành phần.


## 8. Conceptual interpretation questions

1. Vì sao posterior mean, median và MAP là ba quyết định khác nhau dù xuất phát từ cùng một posterior?  
2. Mixture prior mô tả loại bất định nào mà một prior đơn lẻ không mô tả tốt?  
3. Khi dữ liệu đến nhiều, điều gì thường xảy ra với trọng số của các thành phần mixture?  
4. Nếu posterior probability của $x>0.7$ chỉ là 0.657, ta nên diễn giải điều đó ra sao trong ngôn ngữ Bayesian?


## 9. Final takeaway

Lab 6 nhắc rằng Bayesian analysis không chỉ tạo ra posterior, mà còn yêu cầu ta quyết định cách đọc posterior ấy. Khi prior là hỗn hợp, bản thân việc dữ liệu “chọn” thành phần nào cũng là một phần của suy luận.


## 10. Lecture references

Nếu muốn nối từng cụm bài của Lab 6 về đúng lecture trong course, nên đọc lại các phần sau:

- [Bài 2.4: Posterior - Cập nhật Niềm tin với Dữ liệu](../../contents/vi/chapter02/_posts/2025-01-02-02_04_posterior.md): **Bài 1** và **Bài 2**: đọc lại cách đi từ prior và likelihood đến posterior trước khi bàn mean, median, MAP hay xác suất $P(x>0.7\mid y)$.
- [Bài 8.4: Bayesian Decision Analysis - From Inference to Action](../../contents/vi/chapter08/_posts/2025-01-05-08_04_decision_analysis.md): **Bài 1** và **Bài 2**: đọc lại khi muốn hiểu vì sao một posterior có thể được tóm tắt bằng nhiều quyết định khác nhau, và vì sao posterior probability tự nhiên hơn ngôn ngữ “bác bỏ/chấp nhận”.
- [Bài 2.3: Prior - Mã hóa Kiến thức Ban đầu](../../contents/vi/chapter02/_posts/2025-01-02-02_03_prior.md): **Bài 3**, **Bài 4** và **Bài 5**: đọc lại để thấy mixture prior đang mã hóa nhiều kịch bản niềm tin ban đầu cùng lúc, chứ không phải chỉ một họ phân phối duy nhất.
- [Bài 2.5: Prior liên hợp và đại số của cập nhật Bayes](../../contents/vi/chapter02/_posts/2025-01-02-02_05_conjugate_priors.md): **Bài 4** và **Bài 5**: đọc lại khi muốn theo dõi cách từng thành phần Beta hay Gamma trong hỗn hợp vẫn được cập nhật theo quy tắc liên hợp quen thuộc.
- [Bài 11.2: Mixture Models - Clustering & Heterogeneity](../../contents/vi/chapter11/_posts/2025-01-05-11_02_mixture_models.md): **Bài 3** đến **Bài 5**: đặc biệt hữu ích để hiểu trực giác “dữ liệu đang phân bố lại trọng số giữa các kịch bản prior cạnh tranh nhau”.
